<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_L4S_5fold_comparable/L4S_04_unet_5fold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L4S_04 — U-Net + ResNet-34 Segmentación — 5-Fold CV · Dataset Completo (3 799 parches)
**Proyecto:** Detección de Deslizamientos — Landslide4Sense  
**Modelo:** U-Net con encoder ResNet-34 (`segmentation-models-pytorch`)  
**Tarea:** Segmentación pixel-level — máscara binaria 128×128  
**Protocolo:** 5-Fold Stratified CV · Dataset completo

> **Cambios respecto a la versión restringida (notebook 06):**  
> • `N_SUBSET = 3 799` — dataset completo (antes 2 000)  
> • `n_folds = 5` (antes 2)  
> • `epochs = 20` (antes 10)  
> • Umbral óptimo por fold para F1 pixel  
> • Métricas pixel-level: Dice, IoU, F1, AUC-PR  
> • Comparable con: Song et al. 2025, Wang et al. 2024, Ghorbanzadeh 2022

---
### 🔄 Entrenamiento reanudable por fold
Este notebook guarda el estado de cada fold en Drive al terminar. Si la sesión de Colab expira, vuelve a ejecutar todas las celdas — los folds ya completados se **saltan automáticamente** y el entrenamiento continúa desde donde quedó.  
Para correr un fold específico, ajusta `FOLD_TO_RUN` en la **Celda 5**.

In [ ]:
# ── Celda 1: Instalación, Drive y carga de datos ───────────────────────────
from google.colab import drive
import os, sys, h5py, json, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

for pkg in ['segmentation-models-pytorch', 'timm', 'h5py', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

drive.mount('/content/drive')

base_path     = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs      = list(base_path.glob('**/TrainData/img'))
if not img_dirs:
    raise FileNotFoundError('❌ No se encontró TrainData en Drive.')

train_img_dir  = img_dirs[0]
train_mask_dir = train_img_dir.parent / 'mask'
img_list  = sorted(list(train_img_dir.glob('*.h5')))
mask_list = [train_mask_dir / p.name.replace('image_', 'mask_') for p in img_list]
print(f'✅ {len(img_list)} parches detectados')

In [ ]:
# ── Celda 2: Caché de etiquetas de parche ───────────────────────────────────
#
# La etiqueta de parche (positivo/negativo) se usa para estratificar el K-Fold.
# Las máscaras pixel-level se leen durante el entrenamiento.

cache_path = base_path / 'results' / 'labels_cache.json'
cache_path.parent.mkdir(parents=True, exist_ok=True)

if cache_path.exists():
    with open(cache_path) as f:
        cache = json.load(f)
    all_labels = np.array(cache['labels'])
    print(f'✅ Caché cargado: {len(all_labels)} etiquetas')
else:
    print('Generando caché de etiquetas (~4-5 min)...')
    all_labels = []
    for mp in tqdm(mask_list, desc='Leyendo máscaras'):
        with h5py.File(mp, 'r') as f:
            key = list(f.keys())[0]
            all_labels.append(int(f[key][:].max() > 0))
    all_labels = np.array(all_labels)
    with open(cache_path, 'w') as fp:
        json.dump({'labels': all_labels.tolist()}, fp)
    print('✅ Caché guardado')

print(f'Parches positivos: {all_labels.sum()} ({all_labels.mean():.2%}) | negativos: {(1-all_labels).sum()}')
# Dataset completo: N_SUBSET = len(img_list) — sin submuestreo
img_arr  = np.array(img_list)
mask_arr = np.array(mask_list)
print(f'\n✅ Usando dataset COMPLETO: {len(img_arr)} parches')

In [ ]:
# ── Celda 3: Modelo U-Net + ResNet-34 ──────────────────────────────────────
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

print(f'PyTorch: {torch.__version__} | smp: {smp.__version__} | CUDA: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    print('⚠️  Sin GPU. Activa T4 en Runtime → Change runtime type → T4 GPU')

def build_unet(n_channels=14, pretrained=True):
    """U-Net con encoder ResNet-34 preentrenado en ImageNet.
    smp adapta automáticamente la primera capa a n_channels canales."""
    return smp.Unet(
        encoder_name='resnet34',
        encoder_weights='imagenet' if pretrained else None,
        in_channels=n_channels,
        classes=1,
        activation=None,
    )

m_test = build_unet().to(device)
total     = sum(p.numel() for p in m_test.parameters())
trainable = sum(p.numel() for p in m_test.parameters() if p.requires_grad)
print(f'Parámetros totales:    {total:,}')
print(f'Parámetros entrenables: {trainable:,}')
del m_test

In [ ]:
# ── Celda 4: Loss DiceBCE y Dataset de Segmentación ────────────────────────
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth

    def forward(self, logits, targets):
        probs  = torch.sigmoid(logits)
        flat_p = probs.view(-1); flat_t = targets.view(-1)
        return 1 - (2*(flat_p*flat_t).sum() + self.smooth) / (flat_p.sum() + flat_t.sum() + self.smooth)

class DiceBCELoss(nn.Module):
    """Loss combinado óptimo para segmentación de objetos pequeños.
    Dice compensa el desbalance pixel-level (~2-5% positivos dentro del parche)."""
    def __init__(self, pos_weight=None, alpha=0.5):
        super().__init__()
        self.dice  = DiceLoss()
        self.bce   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.alpha = alpha

    def forward(self, logits, targets):
        return self.alpha * self.dice(logits, targets) + (1-self.alpha) * self.bce(logits, targets)

class SegmentationDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        with h5py.File(self.img_paths[idx], 'r') as f:
            key = list(f.keys())[0]
            img  = f[key][()].astype(np.float32)   # (128,128,14)
        with h5py.File(self.mask_paths[idx], 'r') as f:
            key = list(f.keys())[0]
            mask = f[key][()].astype(np.float32)   # (128,128)

        # Clip valores extremos y Z-score por canal
        for c in range(img.shape[2]):
            ch = img[:, :, c]
            p1, p99 = np.percentile(ch, [1, 99])
            ch = np.clip(ch, p1, p99)
            img[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-8)

        img_t  = torch.from_numpy(img.transpose(2, 0, 1))   # (14,128,128)
        mask_t = torch.from_numpy(mask).unsqueeze(0)        # (1,128,128)

        if self.augment:
            if torch.rand(1).item() > 0.5:
                img_t  = torch.flip(img_t,  dims=[2])
                mask_t = torch.flip(mask_t, dims=[2])
            if torch.rand(1).item() > 0.5:
                img_t  = torch.flip(img_t,  dims=[1])
                mask_t = torch.flip(mask_t, dims=[1])

        return img_t, mask_t

## Entrenamiento — 5-Fold CV · 20 épocas · Dataset completo

| Parámetro | Notebook 06 | **Este notebook** |
|-----------|-------------|-------------------|
| `N_SUBSET` | 2 000 | **3 799 (todos)** |
| `n_folds` | 2 | **5** |
| `epochs` | 10 | **20** |
| `patience` | 3 | 5 |

> ⏱ **Tiempo estimado en T4:** ~25–40 min por fold → ~2-3 h total para 5 folds.  
> Los checkpoints se guardan por fold en Drive para poder reanudar si el runtime expira.

### 🔄 Flujo de reanudación
```
Sesión 1:  Fold 1 ✅  →  Fold 2 ✅  →  [runtime expira]
Sesión 2:  Fold 1 ⏭ (skip)  →  Fold 2 ⏭ (skip)  →  Fold 3 ▶️  →  ...
```
Archivos que se guardan por fold en `results/comparable_literatura/unet_5fold/`:
- `fold{N}_best.pt` — pesos del mejor checkpoint
- `fold{N}_results.json` — métricas + historial de épocas
- `fold{N}_vis.npz` — muestras visuales para graficar después

In [ ]:
# ── Celda 5: Configuración y estado de folds ────────────────────────────────
#
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  FOLD_TO_RUN  →  None = correr todos los pendientes                     │
# │               →  1..5 = correr SOLO ese fold (los demás se saltan)      │
# └─────────────────────────────────────────────────────────────────────────┘
FOLD_TO_RUN = None   # <── ajusta aquí antes de correr

CFG = {
    'n_folds'      : 5,
    'epochs'       : 20,
    'batch_size'   : 16,
    'lr'           : 1e-3,
    'pos_weight'   : 0.703,
    'patience'     : 5,
    'seed'         : 42,
    'use_amp'      : True,
    'thr_pixel'    : 0.5,
}

torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])

output_dir = base_path / 'results' / 'comparable_literatura' / 'unet_5fold'
output_dir.mkdir(parents=True, exist_ok=True)

# ── Helpers de estado ──────────────────────────────────────────────────────
def fold_is_done(fold_num):
    """True si el fold ya tiene checkpoint Y resultados guardados en Drive."""
    ckpt = output_dir / f'fold{fold_num}_best.pt'
    res  = output_dir / f'fold{fold_num}_results.json'
    return ckpt.exists() and res.exists()

def load_fold_results(fold_num):
    with open(output_dir / f'fold{fold_num}_results.json') as f:
        return json.load(f)

# ── Pre-calcular splits (determinista gracias a seed fijo) ─────────────────
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_splits = list(skf.split(img_arr, all_labels))  # lista de (train_idx, val_idx)

# ── Resumen de estado actual ───────────────────────────────────────────────
done    = [f for f in range(1, CFG['n_folds']+1) if fold_is_done(f)]
pending = [f for f in range(1, CFG['n_folds']+1) if not fold_is_done(f)]

print('='*55)
print('  ESTADO DE FOLDS')
print('='*55)
for f in range(1, CFG['n_folds']+1):
    estado = '✅ completado' if fold_is_done(f) else '⏳ pendiente'
    print(f'  Fold {f}: {estado}')
print('='*55)

if FOLD_TO_RUN is not None:
    print(f'\n🎯 Modo sesión: correr SOLO Fold {FOLD_TO_RUN}')
    if fold_is_done(FOLD_TO_RUN):
        print(f'   ⚠️  Fold {FOLD_TO_RUN} ya está completado.')
        print(f'      Si quieres re-entrenarlo, borra fold{FOLD_TO_RUN}_best.pt y fold{FOLD_TO_RUN}_results.json de Drive.')
else:
    print(f'\n▶️  Se correrán los folds pendientes: {pending}')
    if not pending:
        print('   🎉 ¡Todos los folds están completos! Ve a la Celda 7 para ver resultados.')

In [ ]:
# ── Celda 6: Entrenamiento — un fold por ejecución o todos los pendientes ───
#
# Cada fold que termina escribe en Drive:
#   fold{N}_best.pt          → pesos del mejor checkpoint
#   fold{N}_results.json     → métricas + historial
#   fold{N}_vis.npz          → muestras visuales (imgs/masks/preds del primer batch val)
#
# Al iniciar, si fold{N}_best.pt + fold{N}_results.json ya existen → se salta.

import torch.optim as optim
from torch.amp import GradScaler, autocast
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                              recall_score, jaccard_score, average_precision_score,
                              precision_recall_curve)

# Determinar qué folds correr en esta sesión
if FOLD_TO_RUN is not None:
    folds_to_run = [FOLD_TO_RUN]
else:
    folds_to_run = [f for f in range(1, CFG['n_folds']+1) if not fold_is_done(f)]

if not folds_to_run:
    print('🎉 No hay folds pendientes. Ejecuta la Celda 7 para ver el resumen.')
else:
    print(f'▶️  Folds a entrenar en esta sesión: {folds_to_run}\n')

for fold_num in folds_to_run:
    fold = fold_num - 1   # índice 0-based para fold_splits
    train_idx, val_idx = fold_splits[fold]

    # ── Skip si ya está completo ────────────────────────────────────────────
    if fold_is_done(fold_num):
        r = load_fold_results(fold_num)
        print(f'⏭  Fold {fold_num} ya completado '
              f'(F1_px={r["f1_pixel_thr05"]:.4f} | Dice={r["dice_thr05"]:.4f} | '
              f'IoU={r["iou_thr05"]:.4f}) — skip')
        continue

    print(f'\n{"="*65}')
    print(f'FOLD {fold_num}/{CFG["n_folds"]}  |  train={len(train_idx)}  val={len(val_idx)}')
    print('='*65)

    model = build_unet(n_channels=14, pretrained=True).to(device)

    train_ds = SegmentationDataset(img_arr[train_idx].tolist(), mask_arr[train_idx].tolist(), augment=True)
    val_ds   = SegmentationDataset(img_arr[val_idx].tolist(),   mask_arr[val_idx].tolist(),   augment=False)
    train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    pos_w     = torch.tensor([CFG['pos_weight']], device=device)
    criterion = DiceBCELoss(pos_weight=pos_w)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CFG['lr'],
        steps_per_epoch=len(train_dl), epochs=CFG['epochs'])
    scaler = GradScaler('cuda', enabled=CFG['use_amp'])

    history = {'train_loss':[], 'val_loss':[], 'val_f1_pixel':[], 'val_dice':[], 'val_iou':[]}
    best_f1, best_epoch, no_improve = 0.0, 0, 0
    t_fold = time.time()

    for epoch in range(1, CFG['epochs'] + 1):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast('cuda', enabled=CFG['use_amp']):
                logits = model(imgs); loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update(); scheduler.step()
            train_loss += loss.item()
        train_loss /= len(train_dl)

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        all_probs_flat, all_mask_flat = [], []

        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(device), masks.to(device)
                with autocast('cuda', enabled=CFG['use_amp']):
                    logits = model(imgs); loss = criterion(logits, masks)
                val_loss += loss.item()
                probs = torch.sigmoid(logits.float()).squeeze(1).cpu().numpy()  # (B,128,128)
                m_np  = masks.squeeze(1).cpu().numpy()
                all_probs_flat.extend(probs.ravel())
                all_mask_flat.extend(m_np.ravel())
        val_loss /= len(val_dl)

        all_probs_flat = np.array(all_probs_flat)
        all_mask_flat  = np.array(all_mask_flat, dtype=np.int32)
        all_preds_flat = (all_probs_flat >= CFG['thr_pixel']).astype(np.int32)

        val_f1_px = f1_score(all_mask_flat, all_preds_flat, zero_division=0)
        tp = (all_preds_flat * all_mask_flat).sum()
        dice = (2*tp) / (all_preds_flat.sum() + all_mask_flat.sum() + 1e-8)
        iou  = jaccard_score(all_mask_flat, all_preds_flat, zero_division=0)

        history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
        history['val_f1_pixel'].append(val_f1_px); history['val_dice'].append(float(dice))
        history['val_iou'].append(float(iou))

        if val_f1_px > best_f1:
            best_f1, best_epoch, no_improve = val_f1_px, epoch, 0
            torch.save(model.state_dict(), output_dir / f'fold{fold_num}_best.pt')
        else:
            no_improve += 1

        print(f'  Ep {epoch:2d}/{CFG["epochs"]} | loss={train_loss:.4f}/{val_loss:.4f} '
              f'| F1_px={val_f1_px:.4f} | Dice={dice:.4f} | IoU={iou:.4f}' +
              (' ★' if no_improve == 0 else ''))
        if no_improve >= CFG['patience']:
            print(f'  ⏹  Early stopping'); break

    # ── Evaluación final con mejor checkpoint ────────────────────────────────
    model.load_state_dict(torch.load(output_dir / f'fold{fold_num}_best.pt', map_location=device))
    model.eval()
    all_probs_flat, all_mask_flat = [], []
    val_imgs_sample, val_masks_sample, val_preds_sample = None, None, None

    with torch.no_grad():
        for batch_i, (imgs, masks) in enumerate(val_dl):
            imgs = imgs.to(device)
            with autocast('cuda', enabled=CFG['use_amp']):
                logits = model(imgs)
            probs = torch.sigmoid(logits.float()).squeeze(1).cpu().numpy()
            m_np  = masks.squeeze(1).numpy()
            all_probs_flat.extend(probs.ravel())
            all_mask_flat.extend(m_np.ravel())
            if batch_i == 0:  # guardar muestras visuales del primer batch
                val_imgs_sample  = imgs.cpu().numpy()[:4]
                val_masks_sample = m_np[:4]
                val_preds_sample = probs[:4]

    all_probs_flat = np.array(all_probs_flat)
    all_mask_flat  = np.array(all_mask_flat, dtype=np.int32)
    all_preds_flat = (all_probs_flat >= CFG['thr_pixel']).astype(np.int32)

    # Umbral óptimo para F1 pixel
    prec_c, rec_c, thr_c = precision_recall_curve(all_mask_flat, all_probs_flat)
    f1_curve  = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-8)
    best_thr  = float(thr_c[np.argmax(f1_curve[:-1])])
    preds_opt = (all_probs_flat >= best_thr).astype(np.int32)

    tp05 = (all_preds_flat * all_mask_flat).sum()
    tp_o = (preds_opt * all_mask_flat).sum()

    fold_result = {
        'fold': fold_num, 'best_epoch': best_epoch, 'best_thr_pixel': best_thr,
        'f1_pixel_thr05':  float(f1_score(all_mask_flat, all_preds_flat, zero_division=0)),
        'f1_pixel_thr_opt':float(f1_score(all_mask_flat, preds_opt,      zero_division=0)),
        'dice_thr05':  float(2*tp05 / (all_preds_flat.sum() + all_mask_flat.sum() + 1e-8)),
        'dice_thr_opt':float(2*tp_o  / (preds_opt.sum()    + all_mask_flat.sum() + 1e-8)),
        'iou_thr05':   float(jaccard_score(all_mask_flat, all_preds_flat, zero_division=0)),
        'iou_thr_opt': float(jaccard_score(all_mask_flat, preds_opt,      zero_division=0)),
        'auc_roc':     float(roc_auc_score(all_mask_flat, all_probs_flat)),
        'auc_pr':      float(average_precision_score(all_mask_flat, all_probs_flat)),
        'history': history,
    }

    # ── Guardar resultados en Drive ──────────────────────────────────────────
    with open(output_dir / f'fold{fold_num}_results.json', 'w') as f:
        json.dump(fold_result, f, indent=2)

    # Guardar muestras visuales como numpy comprimido
    if val_imgs_sample is not None:
        np.savez_compressed(
            output_dir / f'fold{fold_num}_vis.npz',
            imgs=val_imgs_sample,
            masks=val_masks_sample,
            preds=val_preds_sample,
        )

    elapsed = (time.time() - t_fold) / 60
    print(f'\n  ✅ Fold {fold_num} GUARDADO | '
          f'F1_px@0.5={fold_result["f1_pixel_thr05"]:.4f} | '
          f'F1_px@opt={fold_result["f1_pixel_thr_opt"]:.4f} | '
          f'Dice={fold_result["dice_thr05"]:.4f} | '
          f'IoU={fold_result["iou_thr05"]:.4f} | {elapsed:.1f}min')
    print(f'     → fold{fold_num}_best.pt  +  fold{fold_num}_results.json  +  fold{fold_num}_vis.npz')
    del model  # liberar VRAM antes del siguiente fold
    torch.cuda.empty_cache()

# ── Estado final de la sesión ─────────────────────────────────────────────
print()
done_now    = [f for f in range(1, CFG['n_folds']+1) if fold_is_done(f)]
pending_now = [f for f in range(1, CFG['n_folds']+1) if not fold_is_done(f)]
print(f'Folds completados: {done_now}')
if pending_now:
    print(f'Folds pendientes:  {pending_now}  — inicia otra sesión y vuelve a correr desde la Celda 1')
else:
    print('🎉 ¡Todos los folds completados! Ejecuta la Celda 7 para visualización y la Celda 8 para resumen.')

In [ ]:
# ── Celda 7: Visualización de predicciones — carga desde Drive ──────────────
#
# Lee fold{N}_vis.npz. Puedes correr esta celda aunque no hayas entrenado
# en esta sesión, siempre que los archivos estén en Drive.

folds_with_vis = sorted([
    int(p.stem.split('_')[0].replace('fold', ''))
    for p in output_dir.glob('fold*_vis.npz')
])

if not folds_with_vis:
    print('⚠️  No se encontraron archivos fold*_vis.npz en Drive. Entrena al menos un fold primero.')
else:
    # Visualizar el último fold disponible (o cambia el índice)
    vis_fold = folds_with_vis[-1]
    vis_data = np.load(output_dir / f'fold{vis_fold}_vis.npz')
    r_meta   = load_fold_results(vis_fold)

    fig, axes = plt.subplots(4, 3, figsize=(12, 16))
    axes[0][0].set_title('Imagen (RGB falso color)', fontsize=10)
    axes[0][1].set_title('Máscara real', fontsize=10)
    axes[0][2].set_title('Predicción U-Net', fontsize=10)

    for idx in range(min(4, len(vis_data['imgs']))):
        img_np  = vis_data['imgs'][idx]    # (14,128,128)
        mask_np = vis_data['masks'][idx]   # (128,128)
        pred_np = vis_data['preds'][idx]   # (128,128)

        # RGB falso color: B4-B3-B2 (índices 3,2,1)
        rgb = img_np[[3,2,1], :, :].transpose(1,2,0)
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

        axes[idx][0].imshow(rgb); axes[idx][0].axis('off')
        axes[idx][1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1); axes[idx][1].axis('off')
        axes[idx][2].imshow(pred_np, cmap='RdYlGn', vmin=0, vmax=1); axes[idx][2].axis('off')

    plt.suptitle(f'Predicciones U-Net — Fold {vis_fold} | F1_px={r_meta["f1_pixel_thr05"]:.4f}', fontsize=12)
    plt.tight_layout()
    plt.savefig(output_dir / f'fold{vis_fold}_predictions_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Folds con visualización disponible: {folds_with_vis}')

In [ ]:
# ── Celda 8: Resumen final — carga desde Drive ─────────────────────────────
#
# Lee todos los fold{N}_results.json disponibles.
# Puedes correr esta celda en cualquier momento, incluso en una sesión nueva,
# siempre que los archivos estén en Drive. Solo muestra los folds completados.

available_folds = sorted([
    int(p.stem.split('_')[0].replace('fold', ''))
    for p in output_dir.glob('fold*_results.json')
])

if not available_folds:
    print('⚠️  No se encontraron resultados guardados. Entrena al menos un fold primero.')
else:
    fold_results = [load_fold_results(f) for f in available_folds]

    f1_px   = [r['f1_pixel_thr05']   for r in fold_results]
    f1_opt  = [r['f1_pixel_thr_opt'] for r in fold_results]
    dice    = [r['dice_thr05']       for r in fold_results]
    iou     = [r['iou_thr05']        for r in fold_results]
    aucs    = [r['auc_roc']          for r in fold_results]
    auprs   = [r['auc_pr']           for r in fold_results]

    n_done = len(available_folds)
    n_total = CFG['n_folds']
    status_str = f'{n_done}/{n_total} folds' if n_done < n_total else f'5/5 folds (COMPLETO)'

    print('='*70)
    print(f'  RESUMEN — U-Net + ResNet-34 | 5-Fold CV | {status_str}')
    print('='*70)
    print(f'  Folds incluidos: {available_folds}')
    print(f'  F1 pixel @thr=0.5 : {np.mean(f1_px):.4f} ± {np.std(f1_px):.4f}')
    print(f'  F1 pixel @thr_opt : {np.mean(f1_opt):.4f} ± {np.std(f1_opt):.4f}')
    print(f'  Dice  @thr=0.5    : {np.mean(dice):.4f} ± {np.std(dice):.4f}')
    print(f'  IoU   @thr=0.5    : {np.mean(iou):.4f} ± {np.std(iou):.4f}')
    print(f'  AUC-ROC (pixel)   : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
    print(f'  AUC-PR  (pixel)   : {np.mean(auprs):.4f} ± {np.std(auprs):.4f}')
    print('='*70)
    print('  Comparable con:')
    print('  • Song et al. 2025     → Dice ~0.72–0.78 (con módulos atención)')
    print('  • Wang et al. 2024     → IoU  ~0.65–0.71 (encoder dual)')
    print('  • Ghorbanzadeh 2022    → F1   ~0.75–0.82 (official test split)')

    # ── Curvas por fold ────────────────────────────────────────────────────
    n = len(fold_results)
    fig, axes = plt.subplots(n, 3, figsize=(18, 5*n))
    if n == 1: axes = [axes]
    for i, r in enumerate(fold_results):
        h = r['history']; ep = range(1, len(h['train_loss'])+1)
        axes[i][0].plot(ep, h['train_loss'], label='Train'); axes[i][0].plot(ep, h['val_loss'], label='Val')
        axes[i][0].set_title(f'Fold {r["fold"]} — DiceBCE Loss'); axes[i][0].legend(); axes[i][0].grid(alpha=0.3)
        axes[i][1].plot(ep, h['val_f1_pixel'], label='F1 pixel', color='green')
        axes[i][1].plot(ep, h['val_dice'],     label='Dice',     color='steelblue')
        axes[i][1].plot(ep, h['val_iou'],      label='IoU',      color='orange')
        axes[i][1].axvline(r['best_epoch'], color='red', ls='--', alpha=0.5)
        axes[i][1].set_title(f'Fold {r["fold"]} — Métricas pixel'); axes[i][1].legend(); axes[i][1].grid(alpha=0.3)
        axes[i][2].bar(['F1@0.5','F1@opt','Dice','IoU','AUC-ROC','AUC-PR'],
                       [r['f1_pixel_thr05'],r['f1_pixel_thr_opt'],r['dice_thr05'],r['iou_thr05'],r['auc_roc'],r['auc_pr']],
                       color=['#4e8df5','#2ecc71','#e74c3c','#f39c12','#9b59b6','#1abc9c'])
        axes[i][2].set_ylim(0,1); axes[i][2].set_title(f'Fold {r["fold"]} — Resumen métricas'); axes[i][2].grid(alpha=0.3,axis='y')

    plt.suptitle(f'U-Net+ResNet-34 — {status_str} — Dataset Completo (3 799 parches)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_dir / 'training_curves_5fold.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Guardar resumen JSON actualizado ──────────────────────────────────────
    summary = {
        'protocol': {
            'model':'U-Net+ResNet-34', 'n_folds': n_total, 'folds_completed': available_folds,
            'n_samples':len(img_arr), 'epochs':CFG['epochs'], 'seed':CFG['seed'],
            'loss':'DiceBCE(alpha=0.5)',
            'comparable_with':'Song2025,Wang2024,Ghorbanzadeh2022'
        },
        'aggregate': {
            'mean_f1_pixel_thr05':  float(np.mean(f1_px)),   'std_f1_pixel':  float(np.std(f1_px)),
            'mean_f1_pixel_thr_opt':float(np.mean(f1_opt)),  'std_f1_opt':    float(np.std(f1_opt)),
            'mean_dice':            float(np.mean(dice)),     'std_dice':      float(np.std(dice)),
            'mean_iou':             float(np.mean(iou)),      'std_iou':       float(np.std(iou)),
            'mean_auc_roc':         float(np.mean(aucs)),     'std_auc_roc':   float(np.std(aucs)),
            'mean_auc_pr':          float(np.mean(auprs)),    'std_auc_pr':    float(np.std(auprs)),
        },
        'folds': [{k:v for k,v in r.items() if k != 'history'} for r in fold_results]
    }
    with open(output_dir / 'kfold5_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'\n✅ Resumen guardado en: {output_dir}/kfold5_summary.json')
    print(f'   fold{{N}}_best.pt  |  fold{{N}}_results.json  |  kfold5_summary.json')
    print(f'   training_curves_5fold.png  |  fold{{N}}_predictions_sample.png')